# BERT Transfer Learning 전이학습
1. klue/bert-base 베이스모델로 사용
2. NSMC 데이터셋 전이학습
3. 감성분석

## 데이터셋 준비

In [2]:
import tensorflow as tf

ratings_train_path = tf.keras.utils.get_file('ratings_train.txt', 'https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt')
ratings_test_path = tf.keras.utils.get_file('ratings_test.txt', 'https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt')

print(ratings_train_path)
print(ratings_test_path)

14628807/14628807 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4893335/4893335 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
/root/.keras/datasets/ratings_train.txt
/root/.keras/datasets/ratings_test.txt


In [3]:
import pandas as pd

ratings_train_df = pd.read_csv(ratings_train_path, sep='\t')
ratings_train_df

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1
...,...,...,...
149995,6222902,인간이 문제지.. 소는 뭔죄인가..,0
149996,8549745,평점이 너무 낮아서...,1
149997,9311800,이게 뭐요? 한국인은 거들먹거리고 필리핀 혼혈은 착하다?,0
149998,2376369,청춘 영화의 최고봉.방황과 우울했던 날들의 자화상,1


In [4]:
ratings_test_df = pd.read_csv(ratings_test_path, sep='\t')
ratings_test_df

,id,document,label
0,6270596,굳 ㅋ,1
1,9274899,GDNTOPCLASSINTHECLUB,0
2,8544678,뭐야 이 평점들은.... 나쁘진 않지만 10점 짜리는 더더욱 아니잖아,0
3,6825595,지루하지는 않은데 완전 막장임... 돈주고 보기에는....,0
4,6723715,3D만 아니었어도 별 다섯 개 줬을텐데.. 왜 3D로 나와서 제 심기를 불편하게 하죠??,0
...,...,...,...
49995,4608761,오랜만에 평점 로긴했네ㅋㅋ 킹왕짱 쌈뽕한 영화를 만났습니다 강렬하게 육쾌함,1
49996,5308387,의지 박약들이나 하는거다 탈영은 일단 주인공 김대희 닮았고 이등병 찐따 OOOO,0
49997,9072549,그림도 좋고 완성도도 높았지만... 보는 내내 불안하게 만든다,0
49998,5802125,절대 봐서는 안 될 영화.. 재미도 없고 기분만 잡치고.. 한 세트장에서 다 해먹네,0


In [5]:
# 결측치 제거
ratings_train_df.info()
ratings_test_df.info()

ratings_train_df = ratings_train_df.dropna(how='any')
ratings_test_df = ratings_test_df.dropna(how='any')

ratings_train_df.info()
ratings_test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   id        150000 non-null  int64 
 1   document  149995 non-null  object
 2   label     150000 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 3.4+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        50000 non-null  int64 
 1   document  49997 non-null  object
 2   label     50000 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 1.1+ MB
<class 'pandas.core.frame.DataFrame'>
Index: 149995 entries, 0 to 149999
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   id        149995 non-null  int64 
 1   document  149995 non-null  object
 2   label     149995 non-null  int6

In [6]:
# train 1만 5천개, test 5천개의 데이터로 샘플링
ratings_train_df = ratings_train_df.sample(15000, random_state=42)
ratings_test_df = ratings_test_df.sample(5000, random_state=42)

print(ratings_train_df.shape)
print(ratings_test_df.shape)

(15000, 3)
(5000, 3)


In [7]:
# 라벨(긍정/부정) 분포
print(ratings_train_df['label'].value_counts())
print(ratings_test_df['label'].value_counts())

label
1    7541
0    7459
Name: count, dtype: int64
label
0    2516
1    2484
Name: count, dtype: int64


In [8]:
# X, y
X_train = ratings_train_df['document'].values.tolist()
y_train = ratings_train_df['label'].values.tolist()

X_test = ratings_test_df['document'].values.tolist()
y_test = ratings_test_df['label'].values.tolist()

In [9]:
X_train[:5], y_train[:5]

(['원본이 최고',
  '스릴감과 훈훈함이 있는 영화.',
  '굉장히 저평가되는 영화중 하나라고 생각함',
  '정말영화같은이야기 영화여서 영화같은이야기가 좋다',
  '계기도없는데 이상하다'],
 [1, 1, 1, 1, 0])

## 사전 학습 모델 준비
- `klue/bert-base` : 한국어 자연어 이해 데이터셋을 기반으로 공개 된 BERT 계열 모델
- 한국어 문장의 토큰화와 문맥 표현에 적합하므로 한국어 감성 분석 전이 학습 예제로 사용해 본다.
- BERT 본체는 사전 학습 가중치를 사용하고 마지막 분류층은 NSMC 데이터로 새로 학습한다.

In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = 'klue/bert-base'

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

config.json:   0%|          | 0.00/425 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on you

In [10]:
# 데이터 토큰화
X_train_ = tokenizer(X_train, padding=True, truncation=True, return_tensors='pt')
X_test_ = tokenizer(X_test, padding=True, truncation=True, return_tensors='pt')

In [11]:
X_train_.keys()

KeysView({'input_ids': tensor([[    2, 15548,  2052,  ...,     0,     0,     0],
        [    2, 15314,  2434,  ...,     0,     0,     0],
        [    2,  4843,  1535,  ...,     0,     0,     0],
        ...,
        [    2, 10767,  1038,  ...,     0,     0,     0],
        [    2, 11103,  2205,  ...,     0,     0,     0],
        [    2, 24223,  4530,  ...,     0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]])})

In [12]:
# 데이터셋/데이터 로더 준비
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

batch_size = 64

train_dataset = TensorDataset(
    X_train_['input_ids'],
    X_train_['attention_mask'],
    torch.tensor(y_train)
)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

test_dataset = TensorDataset(
    X_test_['input_ids'],
    X_test_['attention_mask'],
    torch.tensor(y_test)
)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


## 모델 학습

In [14]:
from transformers import get_linear_schedule_with_warmup

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# BERT fine-tuning에서는 AdamW를 자주 사용한다.
# AdamW는 weight decay를 gradient update와 분리해서 적용한다.
# 과적합을 줄이고 사전학습 모델을 안정적으로 미세조정하는데 유리하다.
optimizer = optim.AdamW(model.parameters(), lr=1e-5, weight_decay=0.01)

epochs = 5

# 전체 학습 step 수
num_training_steps = len(train_loader) * epochs

# 전체 step의 10% 동안은 학습률을 0에서 설정값까지 서서히 증가 시킨다.
# 사전 학습 모델은 이미 학습 된 가중치를 가지고 있으므로 처음부터 큰 학습률을 적용하면 기존 표현이 급격히 깨질 수 있기 때문이다.
num_warmup_steps = int(num_training_steps * 0.1)

# 초반 10% : 학습률을 선형적으로 증가, 이후 나머지 step : 학습률을 선형적으로 감소
# BERT fine-tuning에서 자주 사용하는 학습률 조절 방식
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps
)

criterion = nn.CrossEntropyLoss()
model = model.to(device)

for epoch in range(epochs):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for input_ids, attention_mask, labels in train_loader:
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        output = model(input_ids, attention_mask)
        logits = output.logits
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_acc = correct / total
    avg_loss = total_loss / len(train_loader)
    print(f'epoch {epoch + 1}/{epochs} : Loss {avg_loss:.6f}, Acc {train_acc:.6f}')

epoch 1/5 : Loss 0.346559, Acc 0.850267
epoch 2/5 : Loss 0.270678, Acc 0.886867
epoch 3/5 : Loss 0.212606, Acc 0.916867
epoch 4/5 : Loss 0.169593, Acc 0.937733
epoch 5/5 : Loss 0.144733, Acc 0.947867


In [15]:
# 데이터 저장

# 모델과 토크나이저 저장
# 파인튜닝이 끝난 모델을 나중에 다시 사용하려면 모델과 토크나어저를 함께 저장해야 한다.
tokenizer.save_pretrained('nsmc_model/klue-bert-base')
# 모델 출력 label id를 사람이 이해할 수 있는 이름으로 매핑한 뒤 저장
model.config.id2label = {0 : '부정', 1 : '긍정'}
model.config.label2id = {'부정' : 0, '긍정' : 1}
model.save_pretrained('nsmc_model/klue-bert-base')


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [16]:
# 저장 된 모델과 토크나이저 로드
local_model_name = 'nsmc_model/klue-bert-base'

tokenizer_ = AutoTokenizer.from_pretrained(local_model_name)
model_ = AutoModelForSequenceClassification.from_pretrained(local_model_name)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [17]:
# 테스트
from transformers import TextClassificationPipeline

pipe = TextClassificationPipeline(
    model=model_,
    tokenizer=tokenizer_
)

pipe('드디어 인생영화를 찾았습니다.')

[{'label': '긍정', 'score': 0.9765773415565491}]

In [23]:
# 허깅페이스 로그인(access token과 같은 값은 코드에 남기지 않도록 유의)
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)

In [25]:
# 허깅페이스에 모델, 토크나이저 push
repo_name = 'blimu/klue-bert-base-nsmc'

model.push_to_hub(repo_name, token=HF_TOKEN)
tokenizer.push_to_hub(repo_name, token=HF_TOKEN)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...jwffopj/model.safetensors:   0%|          |  556kB /  442MB            

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/blimu/klue-bert-base-nsmc/commit/ff6a9932ed9b9949d9b7ad3638b3d4f3c0f1c6a0', commit_message='Upload tokenizer', commit_description='', oid='ff6a9932ed9b9949d9b7ad3638b3d4f3c0f1c6a0', pr_url=None, repo_url=RepoUrl('https://huggingface.co/blimu/klue-bert-base-nsmc', endpoint='https://huggingface.co', repo_type='model', repo_id='blimu/klue-bert-base-nsmc'), pr_revision=None, pr_num=None)

In [27]:
# 허깅페이스 모델 가져와서 사용하기
from transformers import TextClassificationPipeline

pipe = TextClassificationPipeline(
    model=AutoModelForSequenceClassification.from_pretrained(repo_name),
    tokenizer=AutoTokenizer.from_pretrained(repo_name)
)

pipe('드디어 인생영화를 찾았습니다.')

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[{'label': '긍정', 'score': 0.9765773415565491}]

In [28]:
from transformers import pipeline

pipe = pipeline('text-classification', model=repo_name)

reviews = [
    '와 사이다영화~',
    '고구마영화~ 켁켁',
    '눈물이 멈추지 않았어요~ 힐링 제대로 하고 왔습니다.'
]

pipe(reviews)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[{'label': '긍정', 'score': 0.986104428768158},
 {'label': '부정', 'score': 0.544671356678009},
 {'label': '긍정', 'score': 0.991006076335907}]

In [32]:
# 모델 평가
from tqdm import tqdm
import numpy as np

model.eval()
all_preds, all_labels, total_loss = [], [], 0

with torch.no_grad():
  for input_ids, attention_mask, labels in tqdm(test_loader):
    input_ids = input_ids.to(device)
    attention_mask = attention_mask.to(device)
    labels = labels.to(device)

    output = model(input_ids, attention_mask)
    logits = output.logits
    loss = criterion(logits, labels)

    total_loss += loss.item()
    preds = torch.argmax(logits, dim=1)
    all_preds.extend(preds.cpu().numpy())
    all_labels.extend(labels.cpu().numpy())

  avg_loss = total_loss / len(test_loader)
  acc = (np.array(all_preds) == np.array(all_labels)).mean()

  print(f'Evaluation Loss : {avg_loss:.6f} Acc : {acc:.6f}')

100%|██████████| 79/79 [00:33<00:00,  2.34it/s]

Evaluation Loss : 0.372889 Acc : 0.870200
